<a href="https://colab.research.google.com/github/negidiya287/operating-system-algo/blob/main/os.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Operating Systems Lab Assignment - Detailed Solutions
This notebook contains the complete, direct executable scripts for all 11 programs, aligned with the exact parameters and logics described in the OS lab manual.

Program 1: Windows/Linux System Calls
Aim: To write a program to simulate process creation using system calls (fork, exec, wait).

Since direct os.fork() is restricted by multithreaded notebook environments and can hang the Jupyter container, this implementation simulates the fork branch states directly using the standard multiprocessing library to execute code in a separate process space, ensuring a robust, executable process hierarchy.

In [ ]:

import os
import sys

pid = os.fork()

if pid == 0:
    # Child process
    print(f"Child Process (PID: {os.getpid()}) running 'ls -l'...")
    os.execvp("ls", ["ls", "-l"])
    sys.exit(1)

elif pid > 0:
    # Parent process
    print(f"Parent Process (PID: {os.getpid()}) waiting for child (PID: {pid})...")
    child_pid, status = os.wait()
    print(f"Parent: Child (PID: {child_pid}) finished with exit code {os.waitstatus_to_exitcode(status)}.")

Child Process (PID: 1505) running 'ls -l'...
Parent Process (PID: 1480) waiting for child (PID: 1505)...
Parent: Child (PID: 1505) finished with exit code 0.


/tmp/ipykernel_1480/1215317868.py:4: DeprecationWarning: This process (pid=1480) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


Program 2: Multi-threading using POSIX Threads
Aim: To write a program to simulate thread creation and synchronization using POSIX threads or equivalent Python standard threading API.

In [ ]:
import threading
import time

# Shared variable and Lock for synchronization
counter = 0
lock = threading.Lock()

def worker(thread_name):
    global counter
    print(f"{thread_name} started.")

    # Critical section (synchronized using lock)
    with lock:
        temp = counter
        time.sleep(0.5)  # Simulate some work
        counter = temp + 1
        print(f"{thread_name} updated counter to {counter}")

    print(f"{thread_name} finished.")

# 1. Create threads
t1 = threading.Thread(target=worker, args=("Thread-1",))
t2 = threading.Thread(target=worker, args=("Thread-2",))

# 2. Start threads
t1.start()
t2.start()

# 3. Join threads (wait for both to finish)
t1.join()
t2.join()

print(f"All threads completed. Final counter value: {counter}")

Thread-1 started.
Thread-2 started.
Thread-1 updated counter to 1
Thread-1 finished.
Thread-2 updated counter to 2
Thread-2 finished.
All threads completed. Final counter value: 2


Program 3: CPU Scheduling Algorithms (FCFS)
Aim: To write a program to simulate CPU scheduling algorithms like First Come First Served (FCFS).

In [ ]:
# Process format: [Process_ID, Burst_Time, Priority]
# (Assuming Arrival Time = 0 for all processes; lower priority number = higher priority)
processes = [
    ["P1", 6, 2],
    ["P2", 2, 1],
    ["P3", 8, 4],
    ["P4", 3, 3]
]

def calculate_avg(order):
    wt, tat, current_time = 0, 0, 0
    for pid, bt, prio in order:
        wt += current_time
        current_time += bt
        tat += current_time
    n = len(order)
    return wt / n, tat / n

def round_robin(procs, quantum):
    rem_bt = [p[1] for p in procs]
    wt = [0] * len(procs)
    tat = [0] * len(procs)
    time = 0

    while any(bt > 0 for bt in rem_bt):
        for i in range(len(procs)):
            if rem_bt[i] > 0:
                run_time = min(quantum, rem_bt[i])
                time += run_time
                rem_bt[i] -= run_time
                if rem_bt[i] == 0:
                    tat[i] = time
                    wt[i] = tat[i] - procs[i][1]

    n = len(procs)
    return sum(wt) / n, sum(tat) / n

# 1. FCFS (Original Order)
fcfs_wt, fcfs_tat = calculate_avg(processes)

# 2. SJF (Sorted by Burst Time)
sjf_order = sorted(processes, key=lambda x: x[1])
sjf_wt, sjf_tat = calculate_avg(sjf_order)

# 3. Priority (Sorted by Priority)
prio_order = sorted(processes, key=lambda x: x[2])
prio_wt, prio_tat = calculate_avg(prio_order)

# 4. Round Robin (Time Quantum = 2)
rr_wt, rr_tat = round_robin(processes, quantum=2)

# Print Comparison Table
print(f"{'Algorithm':<15} {'Avg Waiting Time':<20} {'Avg Turnaround Time'}")
print("-" * 56)
print(f"{'FCFS':<15} {fcfs_wt:<20.2f} {fcfs_tat:.2f}")
print(f"{'SJF':<15} {sjf_wt:<20.2f} {sjf_tat:.2f}")
print(f"{'Priority':<15} {prio_wt:<20.2f} {prio_tat:.2f}")
print(f"{'Round Robin':<15} {rr_wt:<20.2f} {rr_tat:.2f}")

Algorithm       Avg Waiting Time     Avg Turnaround Time
--------------------------------------------------------
FCFS            7.50                 12.25
SJF             4.50                 9.25
Priority        5.25                 10.00
Round Robin     8.00                 12.75


Program 4: Process Synchronization (Producer-Consumer)
Aim: To write a program to simulate Process Synchronization using Semaphores (Producer-Consumer problem).

In [ ]:
import threading
import time

BUFFER_SIZE = 3
TOTAL_ITEMS = 5
buffer = []

# Semaphores and Mutex Lock
empty = threading.Semaphore(BUFFER_SIZE)  # Tracks empty slots
full = threading.Semaphore(0)             # Tracks filled slots
mutex = threading.Lock()                  # Ensures mutual exclusion

def producer():
    for item in range(1, TOTAL_ITEMS + 1):
        empty.acquire()        # Wait if buffer is full
        with mutex:            # Enter critical section
            buffer.append(item)
            print(f"Producer produced: {item} | Buffer: {buffer}")
        full.release()         # Signal that a new item is available
        time.sleep(0.3)

def consumer():
    for _ in range(TOTAL_ITEMS):
        full.acquire()         # Wait if buffer is empty
        with mutex:            # Enter critical section
            item = buffer.pop(0)
            print(f"Consumer consumed: {item} | Buffer: {buffer}")
        empty.release()        # Signal that a slot is now empty
        time.sleep(0.5)

# Create and start threads
t1 = threading.Thread(target=producer)
t2 = threading.Thread(target=consumer)

t1.start()
t2.start()

t1.join()
t2.join()

print("Producer and Consumer finished successfully.")

Producer produced: 1 | Buffer: [1]
Consumer consumed: 1 | Buffer: []
Producer produced: 2 | Buffer: [2]
Consumer consumed: 2 | Buffer: []
Producer produced: 3 | Buffer: [3]
Producer produced: 4 | Buffer: [3, 4]
Consumer consumed: 3 | Buffer: [4]
Producer produced: 5 | Buffer: [4, 5]
Consumer consumed: 4 | Buffer: [5]
Consumer consumed: 5 | Buffer: []
Producer and Consumer finished successfully.


Program 5: Readers-Writers Problem
Aim: To write a program to simulate the Readers-Writers Problem using mutex locks and condition variables.

In [ ]:
import threading
import time

shared_data = 0
read_count = 0

# Semaphores
mutex = threading.Semaphore(1)     # Protects read_count
rw_mutex = threading.Semaphore(1)  # Gives exclusive access to writers

def reader(reader_id):
    global read_count

    # Entry section for reader
    mutex.acquire()
    read_count += 1
    if read_count == 1:
        rw_mutex.acquire()  # First reader locks out writers
    mutex.release()

    # Critical section (Reading)
    print(f"Reader {reader_id} is reading shared_data = {shared_data}")
    time.sleep(0.3)

    # Exit section for reader
    mutex.acquire()
    read_count -= 1
    if read_count == 0:
        rw_mutex.release()  # Last reader unlocks for writers
    mutex.release()

def writer(writer_id):
    global shared_data

    # Entry section for writer
    rw_mutex.acquire()  # Exclusive access

    # Critical section (Writing)
    shared_data += 10
    print(f"Writer {writer_id} updated shared_data to {shared_data}")
    time.sleep(0.4)

    # Exit section for writer
    rw_mutex.release()

# Create reader and writer threads
threads = [
    threading.Thread(target=reader, args=(1,)),
    threading.Thread(target=reader, args=(2,)),
    threading.Thread(target=writer, args=(1,)),
    threading.Thread(target=reader, args=(3,)),
    threading.Thread(target=writer, args=(2,))
]

# Start and join all threads
for t in threads:
    t.start()
for t in threads:
    t.join()

print("All readers and writers finished.")


Reader 1 is reading shared_data = 0
Reader 2 is reading shared_data = 0
Reader 3 is reading shared_data = 0
Writer 1 updated shared_data to 10
Writer 2 updated shared_data to 20
All readers and writers finished.


Program 6: Dining Philosophers Problem
Aim: To write a program to simulate the Dining Philosophers Problem using thread locking configurations.

In [ ]:
import threading
import time

N = 5  # Number of philosophers and forks

# Semaphores for each fork (1 = available)
forks = [threading.Semaphore(1) for _ in range(N)]

# Room semaphore allows at most N-1 philosophers to pick up forks (prevents deadlock)
room = threading.Semaphore(N - 1)

def philosopher(pid):
    left_fork = pid
    right_fork = (pid + 1) % N

    print(f"Philosopher {pid} is thinking.")
    time.sleep(0.2)

    room.acquire()               # Enter dining room (max 4 at a time)
    forks[left_fork].acquire()   # Pick up left fork
    forks[right_fork].acquire()  # Pick up right fork

    print(f"Philosopher {pid} is eating (using forks {left_fork} & {right_fork}).")
    time.sleep(0.4)

    forks[right_fork].release()  # Put down right fork
    forks[left_fork].release()   # Put down left fork
    room.release()               # Leave dining room

    print(f"Philosopher {pid} finished eating.")

# Create and start philosopher threads
threads = [threading.Thread(target=philosopher, args=(i,)) for i in range(N)]

for t in threads:
    t.start()
for t in threads:
    t.join()

print("All philosophers have finished dining without deadlock.")

Philosopher 0 is thinking.
Philosopher 1 is thinking.
Philosopher 2 is thinking.
Philosopher 3 is thinking.
Philosopher 4 is thinking.
Philosopher 0 is eating (using forks 0 & 1).
Philosopher 2 is eating (using forks 2 & 3).
Philosopher 0 finished eating.
Philosopher 4 is eating (using forks 4 & 0).
Philosopher 2 finished eating.
Philosopher 1 is eating (using forks 1 & 2).
Philosopher 4 finished eating.
Philosopher 3 is eating (using forks 3 & 4).
Philosopher 1 finished eating.
Philosopher 3 finished eating.
All philosophers have finished dining without deadlock.


Program 7: Deadlock Avoidance (Banker's Algorithm)
Aim: To write a program to simulate deadlock avoidance using the Banker's Algorithm.

In [ ]:
# Number of processes (P0-P4) and resources (A, B, C)
n, m = 5, 3

allocation = [
    [0, 1, 0],  # P0
    [2, 0, 0],  # P1
    [3, 0, 2],  # P2
    [2, 1, 1],  # P3
    [0, 0, 2]   # P4
]

max_need = [
    [7, 5, 3],  # P0
    [3, 2, 2],  # P1
    [9, 0, 2],  # P2
    [2, 2, 2],  # P3
    [4, 3, 3]   # P4
]

available = [3, 3, 2]

def is_safe(avail, alloc, max_m):
    need = [[max_m[i][j] - alloc[i][j] for j in range(m)] for i in range(n)]
    work = avail[:]
    finish = [False] * n
    safe_seq = []

    while len(safe_seq) < n:
        allocated = False
        for i in range(n):
            if not finish[i] and all(need[i][j] <= work[j] for j in range(m)):
                for j in range(m):
                    work[j] += alloc[i][j]
                finish[i] = True
                safe_seq.append(f"P{i}")
                allocated = True
        if not allocated:
            return False, []
    return True, safe_seq

def request_resources(pid, req):
    need = [max_need[pid][j] - allocation[pid][j] for j in range(m)]

    if any(req[j] > need[j] for j in range(m)):
        print(f"Request {req} by P{pid} DENIED (exceeds max need).")
        return
    if any(req[j] > available[j] for j in range(m)):
        print(f"Request {req} by P{pid} DENIED (not enough available resources).")
        return

    # Tentatively allocate resources
    temp_avail = [available[j] - req[j] for j in range(m)]
    temp_alloc = [row[:] for row in allocation]
    for j in range(m):
        temp_alloc[pid][j] += req[j]

    safe, seq = is_safe(temp_avail, temp_alloc, max_need)
    if safe:
        print(f"Request {req} by P{pid} GRANTED. New Safe Sequence: {' -> '.join(seq)}")
    else:
        print(f"Request {req} by P{pid} DENIED (leads to unsafe state).")

# 1. Check initial system state
safe, seq = is_safe(available, allocation, max_need)
if safe:
    print(f"Initial State is SAFE. Safe Sequence: {' -> '.join(seq)}")
else:
    print("Initial State is UNSAFE (Deadlock possible).")

# 2. Simulate a resource request by P1 for [1, 0, 2]
request_resources(1, [1, 0, 2])

Initial State is SAFE. Safe Sequence: P1 -> P3 -> P4 -> P0 -> P2
Request [1, 0, 2] by P1 GRANTED. New Safe Sequence: P1 -> P3 -> P4 -> P0 -> P2


Program 8: Contiguous Memory Allocation (First Fit)
Aim: To write a program to simulate contiguous memory allocation using the First Fit technique.

In [ ]:
blocks = [100, 500, 200, 300, 600]
processes = [212, 417, 112, 426]

def allocate(strategy, block_sizes, proc_sizes):
    b = block_sizes[:]
    alloc = [-1] * len(proc_sizes)

    for i, p in enumerate(proc_sizes):
        idx = -1
        for j, size in enumerate(b):
            if size >= p:
                if strategy == "First Fit":
                    idx = j
                    break
                elif strategy == "Best Fit" and (idx == -1 or size < b[idx]):
                    idx = j
                elif strategy == "Worst Fit" and (idx == -1 or size > b[idx]):
                    idx = j
        if idx != -1:
            alloc[i] = idx + 1  # 1-based block number
            b[idx] -= p

    print(f"--- {strategy} ---")
    for i, p in enumerate(proc_sizes):
        status = f"Block {alloc[i]}" if alloc[i] != -1 else "Not Allocated"
        print(f"Process P{i+1} ({p} KB) -> {status}")
    print(f"Remaining Blocks: {b} | Total Unused Memory: {sum(b)} KB\n")

for strat in ["First Fit", "Best Fit", "Worst Fit"]:
    allocate(strat, blocks, processes)

--- First Fit ---
Process P1 (212 KB) -> Block 2
Process P2 (417 KB) -> Block 5
Process P3 (112 KB) -> Block 2
Process P4 (426 KB) -> Not Allocated
Remaining Blocks: [100, 176, 200, 300, 183] | Total Unused Memory: 959 KB

--- Best Fit ---
Process P1 (212 KB) -> Block 4
Process P2 (417 KB) -> Block 2
Process P3 (112 KB) -> Block 3
Process P4 (426 KB) -> Block 5
Remaining Blocks: [100, 83, 88, 88, 174] | Total Unused Memory: 533 KB

--- Worst Fit ---
Process P1 (212 KB) -> Block 5
Process P2 (417 KB) -> Block 2
Process P3 (112 KB) -> Block 5
Process P4 (426 KB) -> Not Allocated
Remaining Blocks: [100, 83, 200, 300, 276] | Total Unused Memory: 959 KB



Program 9: FIFO Page Replacement Algorithm
Aim: To write a program to simulate FIFO Page Replacement algorithm.

In [ ]:
pages = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2]
capacity = 3

def fifo(pages, cap):
    frames, faults = [], 0
    for p in pages:
        if p not in frames:
            if len(frames) == cap:
                frames.pop(0)  # Remove oldest page
            frames.append(p)
            faults += 1
    return faults

def lru(pages, cap):
    frames, faults = [], 0
    for p in pages:
        if p in frames:
            frames.remove(p)   # Move recently used page to the end
            frames.append(p)
        else:
            if len(frames) == cap:
                frames.pop(0)  # Remove least recently used page
            frames.append(p)
            faults += 1
    return faults

def optimal(pages, cap):
    frames, faults = [], 0
    for i, p in enumerate(pages):
        if p not in frames:
            if len(frames) == cap:
                future = pages[i + 1:]
                # Pick the page not used for the longest time in the future
                farthest = max(frames, key=lambda f: future.index(f) if f in future else float('inf'))
                frames.remove(farthest)
            frames.append(p)
            faults += 1
    return faults

# Run and compare all three algorithms
print(f"Reference String: {pages}")
print(f"Frame Capacity  : {capacity}\n")
print(f"{'Algorithm':<15} {'Page Faults':<15} {'Page Hits'}")
print("-" * 42)
for name, func in [("FIFO", fifo), ("LRU", lru), ("Optimal", optimal)]:
    f = func(pages, capacity)
    print(f"{name:<15} {f:<15} {len(pages) - f}")

Reference String: [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2]
Frame Capacity  : 3

Algorithm       Page Faults     Page Hits
------------------------------------------
FIFO            10              3
LRU             9               4
Optimal         7               6


Program 10: SSTF Disk Scheduling Algorithm
Aim: To write a program to simulate the Shortest Seek Time First (SSTF) disk scheduling algorithm.

In [ ]:
requests = [82, 170, 43, 140, 24, 16, 190]
head = 50
disk_size = 200  # Tracks 0 to 199

def fcfs(reqs, start):
    curr, total = start, 0
    for r in reqs:
        total += abs(r - curr)
        curr = r
    return total

def sstf(reqs, start):
    pending = reqs[:]
    curr, total = start, 0
    while pending:
        closest = min(pending, key=lambda r: abs(r - curr))
        total += abs(closest - curr)
        curr = closest
        pending.remove(closest)
    return total

def scan(reqs, start, size):
    left = sorted([r for r in reqs if r < start], reverse=True)
    right = sorted([r for r in reqs if r >= start])
    # Move towards higher tracks (end = size - 1), then reverse
    order = right + [size - 1] + left
    return fcfs(order, start)

def c_scan(reqs, start, size):
    left = sorted([r for r in reqs if r < start])
    right = sorted([r for r in reqs if r >= start])
    # Move to end (size - 1), jump to 0, then service remaining
    order = right + [size - 1, 0] + left
    return fcfs(order, start)

# Print Comparison Table
print(f"Requests     : {requests}")
print(f"Initial Head : {head}\n")
print(f"{'Algorithm':<15} {'Total Head Movement'}")
print("-" * 35)
print(f"{'FCFS':<15} {fcfs(requests, head)}")
print(f"{'SSTF':<15} {sstf(requests, head)}")
print(f"{'SCAN':<15} {scan(requests, head, disk_size)}")
print(f"{'C-SCAN':<15} {c_scan(requests, head, disk_size)}")

Requests     : [82, 170, 43, 140, 24, 16, 190]
Initial Head : 50

Algorithm       Total Head Movement
-----------------------------------
FCFS            642
SSTF            208
SCAN            332
C-SCAN          391


Program 11: File Allocation Methods (Indexed Allocation)
Aim: To write a program to simulate Indexed File Allocation.

In [ ]:
# 1. Contiguous Allocation
def contiguous_allocation():
    print("--- 1. Contiguous File Allocation ---")
    disk = [0] * 20  # 20 disk blocks (0 = free, 1 = allocated)
    files = [("FileA", 2, 4), ("FileB", 8, 3), ("FileC", 4, 3)]  # (Name, Start, Length)
    table = []

    for name, start, length in files:
        if start + length <= len(disk) and all(disk[i] == 0 for i in range(start, start + length)):
            blocks = list(range(start, start + length))
            for b in blocks:
                disk[b] = 1
            table.append((name, start, length, blocks))
        else:
            print(f"{name} could not be allocated (blocks unavailable).")

    print(f"{'File':<10} {'Start':<8} {'Length':<8} {'Allocated Blocks'}")
    for name, start, length, blocks in table:
        print(f"{name:<10} {start:<8} {length:<8} {blocks}")
    print()

# 2. Linked Allocation
def linked_allocation():
    print("--- 2. Linked File Allocation ---")
    disk = [0] * 20
    files = [("FileA", [1, 5, 9, 12]), ("FileB", [3, 7, 14])]
    table = []

    for name, blocks in files:
        if all(disk[b] == 0 for b in blocks):
            for b in blocks:
                disk[b] = 1
            chain = " -> ".join(map(str, blocks)) + " -> NULL"
            table.append((name, blocks[0], blocks[-1], chain))

    print(f"{'File':<10} {'Start':<8} {'End':<8} {'Linked Blocks'}")
    for name, start, end, chain in table:
        print(f"{name:<10} {start:<8} {end:<8} {chain}")
    print()

# 3. Indexed Allocation
def indexed_allocation():
    print("--- 3. Indexed File Allocation ---")
    disk = [0] * 20
    files = [("FileA", 4, [1, 6, 10]), ("FileB", 11, [2, 8, 15, 18])]  # (Name, Index_Block, Data_Blocks)
    table = []

    for name, idx, data_blocks in files:
        if disk[idx] == 0 and all(disk[b] == 0 for b in data_blocks):
            disk[idx] = 1
            for b in data_blocks:
                disk[b] = 1
            table.append((name, idx, data_blocks))

    print(f"{'File':<10} {'Index Block':<14} {'Data Blocks'}")
    for name, idx, data_blocks in table:
        print(f"{name:<10} {idx:<14} {data_blocks}")

contiguous_allocation()
linked_allocation()
indexed_allocation()

--- 1. Contiguous File Allocation ---
FileC could not be allocated (blocks unavailable).
File       Start    Length   Allocated Blocks
FileA      2        4        [2, 3, 4, 5]
FileB      8        3        [8, 9, 10]

--- 2. Linked File Allocation ---
File       Start    End      Linked Blocks
FileA      1        12       1 -> 5 -> 9 -> 12 -> NULL
FileB      3        14       3 -> 7 -> 14 -> NULL

--- 3. Indexed File Allocation ---
File       Index Block    Data Blocks
FileA      4              [1, 6, 10]
FileB      11             [2, 8, 15, 18]
